# Motor Imagery Classification via Gramian Angular Field (GAF) Images + 2D CNN

This notebook transforms 1D EEG signals into 2D **Gramian Angular Summation Field (GASF)** images and trains a 2D CNN classifier.

**Approach (from Kan et al., ICASSP 2021):**
- Each EEG channel is downsampled and converted to polar coordinates
- A pairwise cosine matrix encodes temporal relationships as a 2D image
- 3 EEG channels (C3, Cz, C4) are stacked as a 3-channel image (analogous to RGB)
- A 2D CNN then performs standard image classification

**Data:** BCI Competition IV Graz 2b — 9 subjects, left (0) vs right (1) hand motor imagery  
**Split:** Subjects B01–B07 → Train | Subjects B08–B09 → Test (subject-independent)

## 1. Setup & Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.signal import resample

import tensorflow as tf
from keras.models import Sequential, Model
from keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense,
    Dropout, BatchNormalization, GlobalAveragePooling2D,
    Input
)
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from keras.regularizers import l2

from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_auc_score, roc_curve
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPUs available: {len(tf.config.list_physical_devices('GPU'))}")

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

TensorFlow version: 2.20.0
GPUs available: 0


## 2. Load Data & Train/Test Split

In [ ]:
df = pd.read_pickle("../data/epoched_train.pkl")

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nClass distribution:")
print(df['event_type'].astype(int).value_counts().rename({0: 'Left Hand', 1: 'Right Hand'}))
print(f"\nSubjects: {sorted(df['patient_id'].unique())}")

In [ ]:
# Subject-independent split: B08 and B09 are held out for testing
TEST_SUBJECTS = ['B08', 'B09']

test_df  = df[df['patient_id'].str.startswith(tuple(TEST_SUBJECTS))].copy()
train_df = df[~df['patient_id'].str.startswith(tuple(TEST_SUBJECTS))].copy()

print(f"Training trials : {len(train_df):>5}  (Subjects B01–B07)")
print(f"Testing  trials : {len(test_df):>5}  (Subjects B08–B09)")

## 3. Gramian Angular Field (GAF) Transformation

**Mathematical foundation (Wang & Oates, 2015):**

Given a 1D time series $X = \{x_1, x_2, ..., x_n\}$:

1. **Normalise** to $[-1, 1]$: $\tilde{x}_i = \frac{(x_i - \max X) + (x_i - \min X)}{\max X - \min X}$

2. **Convert to polar coordinates**: $\phi_i = \arccos(\tilde{x}_i)$, $\quad r_i = \frac{i}{N}$

3. **Build GASF matrix** (summation): $G_{i,j} = \cos(\phi_i + \phi_j)$

This preserves **absolute temporal ordering** and encodes **pairwise temporal correlations** in a symmetric 2D matrix. Each cell $(i,j)$ represents the interaction between timepoints $i$ and $j$.

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
EEG_CHANNELS    = ["C3", "Cz", "C4"]   # Motor-relevant channels only
TARGET_POINTS   = 125                   # Downsample 1000 → 125 (8× reduction)
IMAGE_SIZE      = TARGET_POINTS         # Resulting GAF image: 125×125 per channel
N_CHANNELS      = len(EEG_CHANNELS)     # 3 channels → stacked like RGB
# ──────────────────────────────────────────────────────────────────────────────

def compute_gasf(series: np.ndarray) -> np.ndarray:
    """
    Transform a 1D time series into a Gramian Angular Summation Field (GASF) image.

    Steps:
        1. Min-max scale to [-1, 1]
        2. Map to polar angle via arccos
        3. Compute pairwise cosine summation matrix

    Parameters
    ----------
    series : np.ndarray, shape (n_timepoints,)

    Returns
    -------
    gasf : np.ndarray, shape (n_timepoints, n_timepoints)
        Values in [-1, 1]
    """
    # Step 1: Min-max normalisation to [-1, 1]
    x_min, x_max = series.min(), series.max()
    if x_max == x_min:
        # Flat signal edge case — return zero matrix
        return np.zeros((len(series), len(series)))
    scaled = 2.0 * (series - x_min) / (x_max - x_min) - 1.0
    # Clip to [-1, 1] to avoid arccos domain errors from floating-point noise
    scaled = np.clip(scaled, -1.0, 1.0)

    # Step 2: Polar coordinate angles φᵢ = arccos(x̃ᵢ)
    phi = np.arccos(scaled)  # shape: (n,)

    # Step 3: GASF[i,j] = cos(φᵢ + φⱼ)
    # Broadcasting: outer sum of phi with itself
    phi_sum = phi[:, np.newaxis] + phi[np.newaxis, :]  # (n, n)
    gasf = np.cos(phi_sum)

    return gasf.astype(np.float32)


def trial_to_gaf_image(row: pd.Series,
                       channels: list,
                       target_points: int) -> np.ndarray:
    """
    Convert one EEG trial (DataFrame row) into a multi-channel GAF image.

    Returns
    -------
    image : np.ndarray, shape (target_points, target_points, n_channels)
        Each channel corresponds to one EEG electrode's GASF matrix.
    """
    channel_images = []
    for ch in channels:
        signal = np.array(row[ch], dtype=np.float64)
        # Downsample using scipy resample (applies anti-aliasing filter internally)
        signal_ds = resample(signal, target_points)
        gasf = compute_gasf(signal_ds)
        channel_images.append(gasf)

    # Stack along last axis: (H, W, C)
    return np.stack(channel_images, axis=-1)


def build_gaf_dataset(target_df: pd.DataFrame,
                      channels: list,
                      target_points: int,
                      desc: str = "") -> tuple[np.ndarray, np.ndarray]:
    """
    Transform an entire DataFrame of EEG trials into GAF image arrays.

    Returns
    -------
    X : np.ndarray, shape (n_trials, target_points, target_points, n_channels)
    y : np.ndarray, shape (n_trials,)
    """
    n = len(target_df)
    X = np.zeros((n, target_points, target_points, len(channels)), dtype=np.float32)

    for idx, (_, row) in enumerate(target_df.iterrows()):
        if idx % 200 == 0:
            print(f"  {desc}  {idx}/{n} trials processed...", end="\r")
        X[idx] = trial_to_gaf_image(row, channels, target_points)

    y = target_df['event_type'].astype(int).values
    print(f"  {desc}  {n}/{n} trials processed.   ")
    return X, y


print("GAF functions defined.")
print(f"Output image shape per trial: ({IMAGE_SIZE}, {IMAGE_SIZE}, {N_CHANNELS})")

In [ ]:
print("Building GAF training set...")
X_train, y_train = build_gaf_dataset(train_df, EEG_CHANNELS, TARGET_POINTS, desc="[Train]")

print("\nBuilding GAF test set...")
X_test, y_test = build_gaf_dataset(test_df, EEG_CHANNELS, TARGET_POINTS, desc="[Test] ")

print(f"\nX_train shape : {X_train.shape}  (trials × H × W × channels)")
print(f"X_test  shape : {X_test.shape}")
print(f"y_train class distribution — Left: {(y_train==0).sum()}, Right: {(y_train==1).sum()}")
print(f"y_test  class distribution — Left: {(y_test==0).sum()}, Right: {(y_test==1).sum()}")

## 4. Visualise GAF Images

Let's inspect what the transformation produces. The diagonal of a GASF image is the original scaled signal, and off-diagonal entries encode temporal correlations between all pairs of timepoints.

In [ ]:
def plot_gaf_comparison(X: np.ndarray, y: np.ndarray, channels: list):
    """
    Side-by-side GAF image comparison for one Left and one Right trial.
    Rows = classes (Left / Right), Columns = EEG channels (C3 / Cz / C4).
    """
    left_idx  = np.where(y == 0)[0][0]
    right_idx = np.where(y == 1)[0][0]
    samples   = [(left_idx,  "Left Hand (Class 0)",  "Blues_r"),
                 (right_idx, "Right Hand (Class 1)", "Reds_r")]

    fig, axes = plt.subplots(2, len(channels), figsize=(4 * len(channels), 8))
    fig.suptitle("Gramian Angular Summation Fields — Left vs Right Motor Imagery",
                 fontsize=14, fontweight='bold', y=1.01)

    for row_idx, (trial_idx, label, cmap) in enumerate(samples):
        image = X[trial_idx]  # (H, W, C)
        for col_idx, ch_name in enumerate(channels):
            ax = axes[row_idx, col_idx]
            im = ax.imshow(image[:, :, col_idx], cmap=cmap,
                           vmin=-1, vmax=1, interpolation='nearest')
            ax.set_title(f"{ch_name}\n{label}", fontsize=11)
            ax.axis('off')
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()


plot_gaf_comparison(X_train, y_train, EEG_CHANNELS)

In [ ]:
def plot_gaf_rgb_composite(X: np.ndarray, y: np.ndarray):
    """
    Show the 3-channel GAF image as a composite (C3=R, Cz=G, C4=B).
    This is how the CNN sees each trial.
    """
    left_idx  = np.where(y == 0)[0][0]
    right_idx = np.where(y == 1)[0][0]

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    fig.suptitle("3-Channel GAF Composite (C3=R, Cz=G, C4=B)\n"
                 "— as seen by the 2D CNN", fontsize=13, fontweight='bold')

    for ax, idx, title in zip(axes,
                               [left_idx,  right_idx],
                               ["Left Hand (Class 0)", "Right Hand (Class 1)"]):
        # Rescale from [-1,1] to [0,1] for display
        rgb = (X[idx] + 1.0) / 2.0
        ax.imshow(rgb)
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


plot_gaf_rgb_composite(X_train, y_train)

## 5. Preprocessing for the CNN

GAF values are already in **[-1, 1]** by construction, which is a well-conditioned input range for neural networks. We apply a global channel-wise standardisation (mean 0, std 1) computed only on the training set to avoid data leakage.

In [ ]:
# Channel-wise normalisation — fit statistics on training data only
# X shape: (N, H, W, C)  →  normalise across axes (0, 1, 2) per channel

channel_means = X_train.mean(axis=(0, 1, 2), keepdims=True)   # (1, 1, 1, C)
channel_stds  = X_train.std(axis=(0, 1, 2), keepdims=True)    # (1, 1, 1, C)
channel_stds  = np.where(channel_stds == 0, 1.0, channel_stds)  # avoid div/0

X_train_norm = (X_train - channel_means) / channel_stds
X_test_norm  = (X_test  - channel_means) / channel_stds

print(f"X_train_norm — mean: {X_train_norm.mean():.4f}, std: {X_train_norm.std():.4f}")
print(f"X_test_norm  — mean: {X_test_norm.mean():.4f},  std: {X_test_norm.std():.4f}")

## 6. Build the 2D CNN Architecture

```
Input: (125, 125, 3)
  ↓
Block 1: Conv2D(32) → BN → ReLU → Conv2D(32) → BN → ReLU → MaxPool → Dropout
  ↓
Block 2: Conv2D(64) → BN → ReLU → Conv2D(64) → BN → ReLU → MaxPool → Dropout
  ↓
Block 3: Conv2D(128) → BN → ReLU → GlobalAvgPool
  ↓
Dense(128) → ReLU → Dropout → Dense(1) → Sigmoid
```

**Design choices:**
- Double conv blocks before pooling (VGG-style) — effective for capturing fine-grained spatial patterns in GAF textures
- Global Average Pooling instead of Flatten — significantly reduces parameters and improves generalisation on small EEG datasets
- L2 regularisation on conv layers — EEG datasets are relatively small, regularisation prevents overfitting
- Heavy dropout (0.4–0.5) — EEG signals are noisy and subject-dependent

In [ ]:
def build_gaf_cnn(input_shape: tuple, l2_lambda: float = 1e-4) -> Model:
    """
    2D CNN for GAF image classification.

    Parameters
    ----------
    input_shape : tuple  e.g. (125, 125, 3)
    l2_lambda   : float  L2 regularisation strength

    Returns
    -------
    model : compiled Keras Model
    """
    reg = l2(l2_lambda)

    inputs = Input(shape=input_shape, name="gaf_input")

    # ── Block 1 ─────────────────────────────────────────────────────────────
    x = Conv2D(32, (3, 3), padding='same', activation='relu',
               kernel_regularizer=reg, name='conv1a')(inputs)
    x = BatchNormalization(name='bn1a')(x)
    x = Conv2D(32, (3, 3), padding='same', activation='relu',
               kernel_regularizer=reg, name='conv1b')(x)
    x = BatchNormalization(name='bn1b')(x)
    x = MaxPooling2D((2, 2), name='pool1')(x)      # 125 → 62
    x = Dropout(0.3, name='drop1')(x)

    # ── Block 2 ─────────────────────────────────────────────────────────────
    x = Conv2D(64, (3, 3), padding='same', activation='relu',
               kernel_regularizer=reg, name='conv2a')(x)
    x = BatchNormalization(name='bn2a')(x)
    x = Conv2D(64, (3, 3), padding='same', activation='relu',
               kernel_regularizer=reg, name='conv2b')(x)
    x = BatchNormalization(name='bn2b')(x)
    x = MaxPooling2D((2, 2), name='pool2')(x)      # 62 → 31
    x = Dropout(0.3, name='drop2')(x)

    # ── Block 3 ─────────────────────────────────────────────────────────────
    x = Conv2D(128, (3, 3), padding='same', activation='relu',
               kernel_regularizer=reg, name='conv3a')(x)
    x = BatchNormalization(name='bn3a')(x)
    x = Conv2D(128, (3, 3), padding='same', activation='relu',
               kernel_regularizer=reg, name='conv3b')(x)
    x = BatchNormalization(name='bn3b')(x)
    x = MaxPooling2D((2, 2), name='pool3')(x)      # 31 → 15
    x = Dropout(0.4, name='drop3')(x)

    # ── Global Average Pooling + Head ────────────────────────────────────────
    x = GlobalAveragePooling2D(name='gap')(x)       # (N, 128)
    x = Dense(128, activation='relu',
               kernel_regularizer=reg, name='fc1')(x)
    x = Dropout(0.5, name='drop_fc')(x)
    outputs = Dense(1, activation='sigmoid', name='output')(x)

    model = Model(inputs, outputs, name="GAF_CNN")

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.AUC(name='auc')]
    )
    return model


INPUT_SHAPE = (IMAGE_SIZE, IMAGE_SIZE, N_CHANNELS)
model = build_gaf_cnn(INPUT_SHAPE)
model.summary()

## 7. Train the Model

In [ ]:
# ── Callbacks ────────────────────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_auc',
        patience=15,
        restore_best_weights=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

# ── Training ─────────────────────────────────────────────────────────────────
history = model.fit(
    X_train_norm, y_train,
    epochs=80,
    batch_size=32,
    validation_data=(X_test_norm, y_test),
    callbacks=callbacks,
    verbose=1
)

## 8. Evaluate the Model

In [ ]:
# ── Training curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("GAF CNN — Training History", fontsize=14, fontweight='bold')

metrics = [
    ('accuracy', 'val_accuracy', 'Accuracy'),
    ('loss',     'val_loss',     'Binary Cross-Entropy Loss'),
    ('auc',      'val_auc',      'ROC AUC'),
]

for ax, (train_key, val_key, title) in zip(axes, metrics):
    ax.plot(history.history[train_key], label='Train', linewidth=2)
    ax.plot(history.history[val_key],   label='Val',   linewidth=2, linestyle='--')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Epoch")
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Predictions ───────────────────────────────────────────────────────────────
y_pred_probs = model.predict(X_test_norm, verbose=0).flatten()
y_pred       = (y_pred_probs > 0.5).astype(int)

acc     = accuracy_score(y_test, y_pred)
auc     = roc_auc_score(y_test, y_pred_probs)

print("=" * 50)
print("         GAF 2D CNN — Test Set Results")
print("=" * 50)
print(f"  Accuracy : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  ROC AUC  : {auc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['Left Hand', 'Right Hand']))

In [ ]:
# ── Confusion matrix + ROC curve ─────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Left', 'Right'],
            yticklabels=['Left', 'Right'], ax=ax1,
            linewidths=0.5, linecolor='grey')
ax1.set_title("Confusion Matrix — GAF 2D CNN", fontsize=13, fontweight='bold')
ax1.set_xlabel("Predicted", fontsize=11)
ax1.set_ylabel("Actual",    fontsize=11)

# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_pred_probs)
ax2.plot(fpr, tpr, color='steelblue', linewidth=2.5,
         label=f'GAF CNN  (AUC = {auc:.3f})')
ax2.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Random Chance')
ax2.fill_between(fpr, tpr, alpha=0.15, color='steelblue')
ax2.set_xlabel("False Positive Rate", fontsize=11)
ax2.set_ylabel("True Positive Rate",  fontsize=11)
ax2.set_title("ROC Curve — GAF 2D CNN", fontsize=13, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Per-Subject Breakdown

BCI performance varies significantly between subjects. This breakdown reveals which individuals the model generalises to best.

In [ ]:
# Get test indices mapped back to subjects
test_subjects = test_df['patient_id'].values

subject_results = {}
for subject in np.unique(test_subjects):
    mask = test_subjects == subject
    acc_s = accuracy_score(y_test[mask], y_pred[mask])
    n_s   = mask.sum()
    subject_results[subject] = {'accuracy': acc_s, 'n_trials': n_s}

results_df = pd.DataFrame(subject_results).T
results_df['accuracy'] = results_df['accuracy'].round(4)

print("Per-Subject Accuracy (Test Set):")
print(results_df.to_string())

fig, ax = plt.subplots(figsize=(7, 4))
colors = ['#2196F3' if acc >= 0.6 else '#EF5350'
          for acc in results_df['accuracy']]
bars = ax.bar(results_df.index, results_df['accuracy'].astype(float),
              color=colors, edgecolor='white', linewidth=0.7)
ax.axhline(0.5, color='black', linestyle='--', linewidth=1.5, label='Chance (50%)')
ax.axhline(results_df['accuracy'].astype(float).mean(),
           color='orange', linestyle=':', linewidth=2,
           label=f'Mean ({results_df["accuracy"].astype(float).mean():.3f})')
ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy", fontsize=11)
ax.set_title("Per-Subject Accuracy — GAF 2D CNN", fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 10. What the CNN Has Learned — Feature Map Visualisation

Visualising the first conv layer's activations lets us see which spatial regions of the GAF image the network attends to when classifying motor imagery.

In [ ]:
# Build a sub-model that outputs intermediate layer activations
activation_model = Model(
    inputs=model.input,
    outputs=model.get_layer('conv1a').output
)

# Pick one example of each class
left_sample  = X_test_norm[np.where(y_test == 0)[0][0]][np.newaxis, ...]  # (1, H, W, C)
right_sample = X_test_norm[np.where(y_test == 1)[0][0]][np.newaxis, ...]

acts_left  = activation_model.predict(left_sample,  verbose=0)[0]  # (H, W, 32)
acts_right = activation_model.predict(right_sample, verbose=0)[0]

# Display first 8 filters for each class
n_filters = 8
fig, axes = plt.subplots(2, n_filters, figsize=(16, 5))
fig.suptitle("Conv1 Activation Maps — Left (top) vs Right (bottom)",
             fontsize=13, fontweight='bold')

for i in range(n_filters):
    axes[0, i].imshow(acts_left[:, :, i],  cmap='viridis')
    axes[0, i].axis('off')
    axes[0, i].set_title(f"Filter {i+1}", fontsize=8)

    axes[1, i].imshow(acts_right[:, :, i], cmap='viridis')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 11. Save the Model

In [ ]:
model.save("gaf_cnn_motor_imagery.keras")
print("Model saved to gaf_cnn_motor_imagery.keras")

# Save normalisation statistics for inference
np.save("gaf_channel_means.npy", channel_means)
np.save("gaf_channel_stds.npy",  channel_stds)
print("Normalisation statistics saved.")

---
## Summary

| Component | Detail |
|---|---|
| **Transformation** | GASF — 1D signal → 2D cosine pairwise matrix |
| **Channels** | C3, Cz, C4 stacked as 3-channel image |
| **Downsampling** | 1000 → 125 points (8× via `scipy.signal.resample`) |
| **Image size** | 125 × 125 × 3 |
| **Architecture** | 3× double-Conv2D blocks → GlobalAvgPool → Dense |
| **Parameters** | ~400K |
| **Split** | B01–B07 train / B08–B09 test (subject-independent) |

### Potential next steps
- **Band-specific GAFs**: bandpass filter signals to mu (8–13 Hz) and beta (13–30 Hz) bands before transformation, then stack 6 channels (3 channels × 2 bands) — band-specific spatial patterns are the primary motor imagery signature
- **GADF alongside GASF**: compute both summation and difference fields, doubling the input channels for complementary temporal information
- **Data augmentation**: small Gaussian noise injections or time-warping on the 1D signals before GAF transformation
- **Transfer learning**: initialise from ImageNet weights (grayscale → 3-channel GAF is a known effective trick)
- **GT-GAN synthesis** (from the ICASSP paper): generate synthetic GAF training images to augment the limited EEG dataset